<a href="https://colab.research.google.com/github/adalves-ufabc/2026.Q3-PLN/blob/main/2026_Q3_PLN_AULA_15_Notebook_31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Processamento de Linguagem Natural [2026-Q3]**
Prof. Alexandre Donizeti Alves

## ***Word Embeddings***
---



### **BERT**

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Carregar o modelo BERT e o tokenizer
model = BertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Função para obter embedding da frase
def get_embedding(phrase, model, tokenizer):
    inputs = tokenizer(phrase, return_tensors='pt', truncation=True, padding=True)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Definir as disciplinas e a palavra
word = "linguagens formais"
disciplinas = [
    "linguagens formais e autômatos",
    "métodos formais",
    "engenharia de software",
    "comunicação e redes",
    "natureza da informação"
]

# obter o embedding para "linguagens formais"
word_embedding = get_embedding(word, model, tokenizer)

# obter os embeddings para as disciplinas
disciplinas_embeddings = [(disciplina, get_embedding(disciplina, model, tokenizer)) for disciplina in disciplinas]

# calcular a similaridade de cosseno
similarities = []
for disciplina, embedding in disciplinas_embeddings:
    sim = cosine_similarity([word_embedding], [embedding])
    similarities.append((disciplina, sim[0][0]))

# ordenar pela similaridade
similarities.sort(key=lambda x: x[1], reverse=True)

# exibir a disciplina mais similar
print(f"A disciplina mais semelhante à palavra '{word}' é: {similarities[0][0]} com similaridade de {similarities[0][1]:.4f}")


A disciplina mais semelhante à palavra 'linguagens formais' é: linguagens formais e autômatos com similaridade de 0.8140


In [ ]:
similarities

[('linguagens formais e autômatos', np.float32(0.8140334)),
 ('métodos formais', np.float32(0.66957986)),
 ('engenharia de software', np.float32(0.618266)),
 ('natureza da informação', np.float32(0.57010627)),
 ('comunicação e redes', np.float32(0.48914278))]

### **OpenAI**

In [ ]:
!pip install openai

In [ ]:
#@title Definindo a chave da API

from getpass import getpass

OPENAI_API_KEY = getpass()

··········


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI

cliente = OpenAI( api_key= OPENAI_API_KEY )

# função para obter o embedding de uma frase
def get_embedding(text):
    resposta = cliente.embeddings.create(
        model = "text-embedding-3-small",  # modelo de embedding
        input=text
    )
    return np.array(resposta.data[0].embedding)

disciplinas = [
    "linguagens formais e autômatos",
    "métodos formais",
    "engenharia de software",
    "comunicação e redes",
    "natureza da informação"
]

# obter os embeddings para as disciplinas
disciplinas_embeddings = [(disciplina, get_embedding(disciplina)) for disciplina in disciplinas]

In [ ]:
word = "linguagens formais"

# obter o embedding da palavra "engenharia"
word_embedding = get_embedding(word)

# Calcular a similaridade de cosseno entre "engenharia" e as disciplinas
similarities = []
for disciplina, embedding in disciplinas_embeddings:
    sim = cosine_similarity([word_embedding], [embedding])
    similarities.append((disciplina, sim[0][0]))

# Ordenar pela similaridade
similarities.sort(key=lambda x: x[1], reverse=True)

# Exibir a disciplina mais semelhante
print(f"A disciplina mais semelhante à palavra '{word}' é: {similarities[0][0]} com similaridade de {similarities[0][1]:.4f}")

A disciplina mais semelhante à palavra 'linguagens formais' é: linguagens formais e autômatos com similaridade de 0.8294


In [ ]:
similarities

[('linguagens formais e autômatos', np.float64(0.8293966004602246)),
 ('métodos formais', np.float64(0.6911656825752346)),
 ('comunicação e redes', np.float64(0.44385935391666775)),
 ('engenharia de software', np.float64(0.43948278347168374)),
 ('natureza da informação', np.float64(0.40339428830400975))]

### **Cohere**

In [ ]:
!pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 26.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cohere]


In [ ]:
from getpass import getpass

COHERE_API_KEY = getpass()

··········


In [ ]:
import cohere
from sklearn.metrics.pairwise import cosine_similarity

# crie uma instância do cliente Cohere
co = cohere.Client(COHERE_API_KEY)

# função para obter embeddings (vetores) usando Cohere
def get_embedding(text):
    # É necessário especificar o modelo e o input_type nas versões mais recentes da API
    response = co.embed(
        texts=[text],
        model='embed-v5.0-pro',
        input_type='search_document'
    )
    return response.embeddings[0]

disciplinas = [
    "linguagens formais e autômatos",
    "métodos formais",
    "engenharia de software",
    "comunicação e redes",
    "natureza da informação"
]

# obter os embeddings para a palavra e as disciplinas
disciplinas_embeddings = [(disciplina, get_embedding(disciplina)) for disciplina in disciplinas]

In [ ]:
word = "linguagens formais"

word_embedding = get_embedding(word)

# calcular a similaridade de cosseno entre "engenharia" e as disciplinas
similarities = []
for disciplina, embedding in disciplinas_embeddings:
    sim = cosine_similarity([word_embedding], [embedding])
    similarities.append((disciplina, sim[0][0]))

# ordenar as disciplinas por similaridade
similarities.sort(key=lambda x: x[1], reverse=True)

# exibir a disciplina mais semelhante
print(f"A disciplina mais semelhante à palavra '{word}' é: {similarities[0][0]} com similaridade de {similarities[0][1]:.4f}")


A disciplina mais semelhante à palavra 'linguagens formais' é: linguagens formais e autômatos com similaridade de 0.9018


In [ ]:
similarities

[('linguagens formais e autômatos', np.float64(0.9017772871543883)),
 ('métodos formais', np.float64(0.8941451618272631)),
 ('engenharia de software', np.float64(0.7909146074289588)),
 ('natureza da informação', np.float64(0.7736161444993186)),
 ('comunicação e redes', np.float64(0.7600644009550779))]

### **Gemini**

In [ ]:
!pip install google-genai

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

from google import genai

cliente = genai.Client( api_key= GEMINI_API_KEY)

# função para obter o embedding de uma frase
def get_embedding(text):
    resposta = cliente.models.embed_content(
        model = "gemini-embedding-001",  # modelo de embedding
        contents=[text]
    )
    # Extract the embedding values and convert to a numpy array
    return np.array(resposta.embeddings[0].values)

disciplinas = [
    "linguagens formais e autômatos",
    "métodos formais",
    "engenharia de software",
    "comunicação e redes",
    "natureza da informação"
]

# obter os embeddings para as disciplinas
disciplinas_embeddings = [(disciplina, get_embedding(disciplina)) for disciplina in disciplinas]

In [ ]:
word = "linguagens formais"

word_embedding = get_embedding(word)

# calcular a similaridade de cosseno entre "engenharia" e as disciplinas
similarities = []
for disciplina, embedding in disciplinas_embeddings:
    sim = cosine_similarity([word_embedding], [embedding])
    similarities.append((disciplina, sim[0][0]))

# ordenar as disciplinas por similaridade
similarities.sort(key=lambda x: x[1], reverse=True)

# exibir a disciplina mais semelhante
print(f"A disciplina mais semelhante à palavra '{word}' é: {similarities[0][0]} com similaridade de {similarities[0][1]:.4f}")

A disciplina mais semelhante à palavra 'linguagens formais' é: linguagens formais e autômatos com similaridade de 0.8668


In [ ]:
similarities

[('linguagens formais e autômatos', np.float64(0.8667903857582377)),
 ('métodos formais', np.float64(0.8178577104287312)),
 ('comunicação e redes', np.float64(0.664057558118983)),
 ('engenharia de software', np.float64(0.6617742780616254)),
 ('natureza da informação', np.float64(0.6350558829856818))]

### **Referências**

https://developers.openai.com/api/docs/guides/embeddings

https://docs.cloud.google.com/gemini-enterprise-agent-platform/reference/models/text-embeddings-api

https://docs.cohere.com/v2/docs/semantic-search-embed